In [2]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

# -------------------------------
# SETUP
# -------------------------------

BASE_URL = "https://nreganarep.nic.in/"
BASE_FOLDER = "MGNREGA_DATA"

os.makedirs(BASE_FOLDER, exist_ok=True)

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 30)

print("Starting scraper...")

# -------------------------------
# FUNCTION: Solve Captcha + Reach Dropdown
# -------------------------------

def go_to_dropdown_page():
    driver.get(BASE_URL)
    time.sleep(2)

    # Solve captcha
    captcha_value = driver.find_element(
        By.ID, "ContentPlaceHolder1_hfCaptcha"
    ).get_attribute("value")

    driver.find_element(
        By.ID, "ContentPlaceHolder1_txtCaptcha"
    ).send_keys(captcha_value)

    driver.find_element(
        By.ID, "ContentPlaceHolder1_btnLogin"
    ).click()

    time.sleep(3)

# -------------------------------
# FUNCTION: Extract Table Data
# -------------------------------

def extract_progress_report(year, state_name):

    soup = BeautifulSoup(driver.page_source, "html.parser")
    tables = soup.find_all("table")

    if not tables:
        print("   No tables found.")
        return None

    # Choose largest table (district table)
    largest_table = max(tables, key=lambda t: len(t.find_all("tr")))

    rows = largest_table.find_all("tr")
    data = []

    for row in rows:
        cols = row.find_all("td")
        cols = [col.get_text(strip=True) for col in cols]
        if len(cols) > 5:
            data.append(cols)

    df = pd.DataFrame(data)

    if df.shape[1] < 19:
        print("   Unexpected table format.")
        return None

    # Select required columns by position
    selected_df = df.iloc[:, [1, 2, 3, 10, 15, 18]].copy()

    selected_df.columns = [
        "District",
        "Unskilled_Wage_Current",
        "Unskilled_Wage_Previous_Liability",
        "Total_Expenditure",
        "Avg_Wage_Per_Personday",
        "Cost Per Personday"
    ]

    # Remove TOTAL row
    selected_df = selected_df[
        selected_df["District"].str.upper() != "TOTAL"
    ]

    # Drop 2nd & 3rd row
    selected_df = selected_df.drop(index=[1, 2], errors="ignore")

    selected_df["Year"] = year
    selected_df["State"] = state_name

    selected_df.reset_index(drop=True, inplace=True)

    return selected_df

# -------------------------------
# GET AVAILABLE YEARS
# -------------------------------

go_to_dropdown_page()

year_dropdown = Select(
    driver.find_element(By.ID, "ContentPlaceHolder1_ddlfinyr")
)

years = [
    option.text.strip()
    for option in year_dropdown.options
    if option.text.strip() not in ["Select", ""]
]

# Optional: process from oldest to newest
years = sorted(years)
years = [year for year in years if year >= "2014-15"]

print("Years found:", years)

# -------------------------------
# MAIN LOOP
# -------------------------------

for year in years:

    print(f"\nProcessing Year: {year}")

    # Create year folder
    year_folder = os.path.join(BASE_FOLDER, year)
    os.makedirs(year_folder, exist_ok=True)

    # Restart page for this year
    go_to_dropdown_page()

    # Select year
    Select(driver.find_element(
        By.ID, "ContentPlaceHolder1_ddlfinyr"
    )).select_by_visible_text(year)

    time.sleep(2)

    # Get states
    state_dropdown = Select(
        driver.find_element(By.ID, "ContentPlaceHolder1_ddl_States")
    )

    states = [
        option.text.strip()
        for option in state_dropdown.options
        if option.text.strip() not in ["Select", "---Select---", "ALL", ""]
    ]

    for state_name in states:

        print(f"   State: {state_name}")

        try:
            # Restart page fresh for each state
            go_to_dropdown_page()

            Select(driver.find_element(
                By.ID, "ContentPlaceHolder1_ddlfinyr"
            )).select_by_visible_text(year)

            time.sleep(2)

            Select(driver.find_element(
                By.ID, "ContentPlaceHolder1_ddl_States"
            )).select_by_visible_text(state_name)

            time.sleep(3)

            # Click Outlays & Outcomes link
            progress_link = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//a[contains(., 'Outlays & Outcomes')]")
                )
            )

            driver.execute_script("arguments[0].click();", progress_link)

            time.sleep(3)

            # Extract table
            df = extract_progress_report(year, state_name)

            if df is None:
                print("   Skipped due to extraction issue.")
                continue

            # Save CSV
            file_path = os.path.join(year_folder, f"{state_name}.csv")
            df.to_csv(file_path, index=False)

            print(f"   Saved: {file_path}")

            time.sleep(1)

        except Exception as e:
            print(f"   Error processing {state_name}: {e}")
            continue

print("\nScraping completed successfully.")
driver.quit()

Starting scraper...
Years found: ['2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']

Processing Year: 2014-2015
   State: ANDAMAN AND NICOBAR
   Saved: MGNREGA_DATA\2014-2015\ANDAMAN AND NICOBAR.csv
   State: ANDHRA PRADESH
   Saved: MGNREGA_DATA\2014-2015\ANDHRA PRADESH.csv
   State: ARUNACHAL PRADESH
   Saved: MGNREGA_DATA\2014-2015\ARUNACHAL PRADESH.csv
   State: ASSAM
   Saved: MGNREGA_DATA\2014-2015\ASSAM.csv
   State: BIHAR
   No tables found.
   Skipped due to extraction issue.
   State: CHHATTISGARH
   Saved: MGNREGA_DATA\2014-2015\CHHATTISGARH.csv
   State: DN HAVELI AND DD
   Saved: MGNREGA_DATA\2014-2015\DN HAVELI AND DD.csv
   State: GOA
   Saved: MGNREGA_DATA\2014-2015\GOA.csv
   State: GUJARAT
   Saved: MGNREGA_DATA\2014-2015\GUJARAT.csv
   State: HARYANA
   Saved: MGNREGA_DATA\2014-2015\HARYANA.csv
   State: HIMACHAL PRADESH
   Saved: MGNREGA_DATA\2014-2015\HIMACHAL 

In [7]:
import os

BASE_FOLDER = "MGNREGA_DATA"

# Expected years (same rule you used)
years = sorted(os.listdir(BASE_FOLDER))

# Expected states (full list)
states = [
"ANDAMAN AND NICOBAR","ANDHRA PRADESH","ARUNACHAL PRADESH","ASSAM",
"BIHAR","CHHATTISGARH","GOA","GUJARAT","HARYANA","HIMACHAL PRADESH",
"JAMMU AND KASHMIR","JHARKHAND","KARNATAKA","KERALA","LADAKH",
"MADHYA PRADESH","MAHARASHTRA","MANIPUR","MEGHALAYA","MIZORAM",
"NAGALAND","ODISHA","PUNJAB","RAJASTHAN","SIKKIM","TAMIL NADU",
"TELANGANA","TRIPURA","UTTAR PRADESH","UTTARAKHAND","WEST BENGAL"
]

missing_jobs = []

for year in years:

    year_path = os.path.join(BASE_FOLDER, year)

    for state in states:

        file_path = os.path.join(year_path, f"{state}.csv")

        if not os.path.exists(file_path):
            missing_jobs.append((year, state))


print("Total missing:", len(missing_jobs))

for job in missing_jobs:
    print(job)

# Save missing list
with open("missing_jobs.txt", "w") as f:
    for year, state in missing_jobs:
        f.write(f"{year},{state}\n")

Total missing: 0


In [8]:
missing_jobs = []

with open("missing_jobs.txt") as f:
    for line in f:
        year, state = line.strip().split(",")
        missing_jobs.append((year, state))

print("Jobs to recover:", len(missing_jobs))

Jobs to recover: 0


In [6]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE_URL = "https://nreganarep.nic.in/"
BASE_FOLDER = "MGNREGA_DATA"

# -------------------------------
# FUNCTION: Solve Captcha
# -------------------------------

def go_to_dropdown_page(driver):

    driver.get(BASE_URL)
    time.sleep(2)

    captcha_value = driver.find_element(
        By.ID, "ContentPlaceHolder1_hfCaptcha"
    ).get_attribute("value")

    driver.find_element(
        By.ID, "ContentPlaceHolder1_txtCaptcha"
    ).send_keys(captcha_value)

    driver.find_element(
        By.ID, "ContentPlaceHolder1_btnLogin"
    ).click()

    time.sleep(3)

# -------------------------------
# FUNCTION: Extract Table
# -------------------------------

def extract_progress_report(driver, year, state_name):

    soup = BeautifulSoup(driver.page_source, "html.parser")
    tables = soup.find_all("table")

    if not tables:
        print("   No tables found.")
        return None

    largest_table = max(tables, key=lambda t: len(t.find_all("tr")))

    rows = largest_table.find_all("tr")
    data = []

    for row in rows:
        cols = row.find_all("td")
        cols = [col.get_text(strip=True) for col in cols]

        if len(cols) > 5:
            data.append(cols)

    df = pd.DataFrame(data)

    if df.shape[1] < 19:
        print("   Unexpected table format:", df.shape)
        return None

    selected_df = df.iloc[:, [1, 2, 3, 10, 15, 18]].copy()

    selected_df.columns = [
        "District",
        "Unskilled_Wage_Current",
        "Unskilled_Wage_Previous_Liability",
        "Total_Expenditure",
        "Avg_Wage_Per_Personday",
        "Cost Per Personday"
    ]

    selected_df = selected_df[
        selected_df["District"].str.upper() != "TOTAL"
    ]

    selected_df = selected_df.drop(index=[1,2], errors="ignore")

    selected_df["Year"] = year
    selected_df["State"] = state_name

    selected_df.reset_index(drop=True, inplace=True)

    return selected_df

# -------------------------------
# LOAD MISSING JOBS
# -------------------------------

missing_jobs = []

with open("missing_jobs.txt") as f:
    for line in f:
        year, state = line.strip().split(",")
        missing_jobs.append((year, state))

print("Jobs to recover:", len(missing_jobs))

# -------------------------------
# RECOVERY LOOP
# -------------------------------

for year, state_name in missing_jobs:

    print(f"\nRecovering: {year} - {state_name}")

    try:

        driver = webdriver.Chrome()
        wait = WebDriverWait(driver, 30)

        go_to_dropdown_page(driver)

        # Select year
        Select(driver.find_element(
            By.ID, "ContentPlaceHolder1_ddlfinyr"
        )).select_by_visible_text(year)

        time.sleep(2)

        # Select state
        Select(driver.find_element(
            By.ID, "ContentPlaceHolder1_ddl_States"
        )).select_by_visible_text(state_name)

        time.sleep(3)

        # Click Outlays & Outcomes
        progress_link = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//a[contains(., 'Outlays & Outcomes')]")
            )
        )

        driver.execute_script("arguments[0].click();", progress_link)

        time.sleep(3)

        # Extract table
        df = extract_progress_report(driver, year, state_name)

        if df is None:
            print("   Still failed.")
            driver.quit()
            continue

        year_folder = os.path.join(BASE_FOLDER, year)
        os.makedirs(year_folder, exist_ok=True)

        file_path = os.path.join(year_folder, f"{state_name}.csv")

        df.to_csv(file_path, index=False)

        print("   Recovered successfully.")

        driver.quit()

        time.sleep(2)

    except Exception as e:

        print(f"   Error recovering {year} - {state_name}: {e}")

        try:
            driver.quit()
        except:
            pass

print("\nRecovery run completed.")

Jobs to recover: 25

Recovering: 2014-2015 - BIHAR
   Recovered successfully.

Recovering: 2014-2015 - JAMMU AND KASHMIR
   Recovered successfully.

Recovering: 2014-2015 - LADAKH
   Recovered successfully.

Recovering: 2014-2015 - WEST BENGAL
   Recovered successfully.

Recovering: 2015-2016 - BIHAR
   Recovered successfully.

Recovering: 2015-2016 - JAMMU AND KASHMIR
   Recovered successfully.

Recovering: 2015-2016 - LADAKH
   Recovered successfully.

Recovering: 2015-2016 - WEST BENGAL
   Recovered successfully.

Recovering: 2016-2017 - BIHAR
   Recovered successfully.

Recovering: 2016-2017 - JAMMU AND KASHMIR
   Recovered successfully.

Recovering: 2016-2017 - LADAKH
   Recovered successfully.

Recovering: 2016-2017 - WEST BENGAL
   Recovered successfully.

Recovering: 2017-2018 - BIHAR
   Recovered successfully.

Recovering: 2017-2018 - JAMMU AND KASHMIR
   Recovered successfully.

Recovering: 2017-2018 - LADAKH
   Recovered successfully.

Recovering: 2017-2018 - WEST BENGAL
   